In [ ]:
import os
from pathlib import Path

DATA_DIR = Path("data")

def load_documents(data_dir=DATA_DIR):
    docs = []
    for path in data_dir.glob("*.txt"):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        docs.append({"filename": path.name, "text": text})
    return docs

docs = load_documents()
len(docs), docs[0]["filename"]


In [ ]:
def chunk_text(text, max_chars=800):
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]
        chunks.append(chunk)
        start = end
    return chunks

chunks = []
for doc in docs:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "source": doc["filename"],
            "text": chunk
        })

len(chunks), chunks[0]

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast model

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = model.encode(chunk_texts, convert_to_numpy=True)
chunk_embeddings.shape

In [ ]:
def retrieve_relevant_chunks(query, top_k=5):
    query_emb = model.encode([query], convert_to_numpy=True)
    sims = cosine_similarity(query_emb, chunk_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in top_idx:
        results.append({
            "score": float(sims[idx]),
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"]
        })
    return results

results = retrieve_relevant_chunks("What is the DTP3 T 331 used for?")
results[0]

In [ ]:
def answer_question(query, top_k=3):
    results = retrieve_relevant_chunks(query, top_k=top_k)
    print(f"Question: {query}\n")
    print("Most relevant information:\n")
    for r in results:
        print(f"Source: {r['source']} (score: {r['score']:.3f})")
        print(r["text"])
        print("-" * 80)

answer_question("What are the main features of the DTP3 T 331?")

In [ ]:
def build_context(results):
    context = ""
    for r in results:
        context += f"From {r['source']}:\n{r['text']}\n\n"
    return context

def build_prompt(query, results):
    context = build_context(results)
    prompt = f"""You are an assistant answering questions about Extron products.

Use ONLY the information in the context below. If the answer is not there, say you don't know.

Context:
{context}

Question: {query}
Answer:"""
    return prompt